# Resgatécnica — Benchmark LLM Local com A100 (Qwen2.5-32B)

**Objetivo:** Testar se um LLM local de 32B na A100 consegue igualar a qualidade do Claude Haiku.

**Configuração:**
- Modelo: `qwen2.5:32b` Q4_K_M (~18 GB VRAM)
- GPU: A100 40GB ou 80GB (~15 tok/s esperado)
- Velocidade: ~2× mais rápido que o Qwen2.5-14B na T4

**Sequência:**
1. Verificar GPU A100
2. Upload do ZIP com os dados
3. Instalar Ollama + baixar modelo (~18 GB, ~15 min)
4. Executar agente
5. Telemetria e comparação com Haiku
6. Download dos resultados

> **Antes de começar:** `Ambiente de execução → Alterar tipo → GPU A100`

## Célula 1 — Verificar GPU A100

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free'],
    capture_output=True, text=True, check=False
)
if result.returncode == 0:
    info = result.stdout.strip()
    print('GPU detectada:', info)
    if 'A100' not in info:
        print()
        print('⚠ ATENÇÃO: GPU não é A100.')
        print('  Vá em Ambiente de execução → Alterar tipo de ambiente → A100.')
        print('  Com T4 ou L4, o modelo de 32B ficará muito lento (~3 tok/s).')
    else:
        print('✓ A100 confirmada — pronto para qwen2.5:32b.')
else:
    print('AVISO: GPU não detectada. Altere o tipo de ambiente para A100.')

## Célula 2 — Upload do ZIP

In [ ]:
from google.colab import files
import os

print('Faça upload do arquivo resgatecnica_colab.zip')
uploaded = files.upload()

zip_name = [k for k in uploaded.keys() if k.endswith('.zip')]
if not zip_name:
    raise RuntimeError('Nenhum arquivo .zip encontrado no upload.')
zip_name = zip_name[0]
print(f'Upload recebido: {zip_name} ({len(uploaded[zip_name]):,} bytes)')

## Célula 3 — Extrair ZIP e verificar estrutura

In [ ]:
import zipfile, pathlib

WORK_DIR = pathlib.Path('/content/resgatecnica')
WORK_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(WORK_DIR)

checks = {
    'pncp_agente.py':       WORK_DIR / 'pncp_agente.py',
    'pncp_licitacoes.json': WORK_DIR / 'pncp_licitacoes.json',
    'exclusoes.yaml':       WORK_DIR / 'exclusoes.yaml',
    'prompts/':             WORK_DIR / 'prompts',
    'editais/':             WORK_DIR / 'editais',
}
for nome, caminho in checks.items():
    status = '✓' if caminho.exists() else '✗ FALTANDO'
    print(f'  {status}  {nome}')

n_editais = sum(1 for d in (WORK_DIR / 'editais').iterdir()
                if d.is_dir() and (d / 'itens.json').exists())
print(f'\nLicitações com itens.json: {n_editais}')
print(f'Diretório de trabalho: {WORK_DIR}')

## Célula 4 — Instalar dependências Python

In [ ]:
!pip install jsonschema requests pyyaml -q
print('Dependências instaladas.')

## Célula 5 — Instalar Ollama e iniciar servidor

O Ollama detecta automaticamente a A100 via CUDA.

In [ ]:
import subprocess, time, requests as req, os, sys

# zstd é dependência do script de instalação do Ollama
print('Instalando zstd...')
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['sudo', 'apt-get', 'install', 'zstd', '-y'], capture_output=True)
print('zstd OK.')

print('Instalando Ollama...')
for tentativa in range(1, 4):
    result = subprocess.run(
        'curl -fsSL --retry 3 --retry-delay 5 https://ollama.com/install.sh | sh',
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'Ollama instalado (tentativa {tentativa}).')
        break
    print(f'Tentativa {tentativa}/3 falhou — retentando em 5s...')
    print('  STDERR:', result.stderr[-200:])
    time.sleep(5)
else:
    print('STDERR completo:', result.stderr[-1000:])
    sys.exit('Falha na instalação após 3 tentativas. Re-execute esta célula.')

OLLAMA_BIN = '/usr/local/bin/ollama'
if not os.path.exists(OLLAMA_BIN):
    sys.exit(f'Binário não encontrado: {OLLAMA_BIN}')
print(f'Ollama instalado: {OLLAMA_BIN}')

# -----------------------------------------------------------------------
# CRÍTICO: apontar Ollama para as bibliotecas CUDA do Colab.
# Sem isso o modelo roda na CPU (~0.3 tok/s) em vez da GPU (~15 tok/s).
# -----------------------------------------------------------------------
env = dict(os.environ)
env['OLLAMA_HOST']          = '0.0.0.0'
env['CUDA_VISIBLE_DEVICES'] = '0'
env['LD_LIBRARY_PATH']      = '/usr/lib64-nvidia:' + env.get('LD_LIBRARY_PATH', '')

print('Iniciando ollama serve...')
srv = subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=env
)

for i in range(15):
    time.sleep(2)
    try:
        r = req.get('http://localhost:11434/api/tags', timeout=3)
        if r.status_code == 200:
            print(f'Ollama respondendo (tentativa {i+1}). Pronto.')
            break
    except Exception:
        print(f'  Aguardando servidor... ({i+1}/15)')
else:
    sys.exit('Servidor Ollama não iniciou após 30s.')

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', gpu.stdout.strip())

## Célula 6 — Baixar modelo Qwen2.5-32B

**Download: ~18 GB** — aguarde 15-25 minutos.  
O warm-up carrega o modelo na VRAM da A100 e mede a velocidade real.

In [ ]:
import time, requests as req, subprocess, sys

MODELO = 'qwen2.5:32b'   # 32B Q4_K_M ≈ 18 GB VRAM — cabe na A100 40GB e 80GB

def vram_info():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    return r.stdout.strip()

print(f'Baixando {MODELO}... (~18 GB, pode levar 15-25 min)')
t0 = time.time()
result = subprocess.run([OLLAMA_BIN, 'pull', MODELO], capture_output=True, text=True)
if result.returncode != 0:
    print('ERRO no download:', result.stderr[-500:])
    sys.exit('Falha no download.')
print(f'Download concluído em {(time.time()-t0)/60:.1f} min.')
print('VRAM antes do warm-up:', vram_info())

# --- Warm-up 1: carrega modelo na VRAM (cold start ~200-300s para 32B) ---
print(f'\nWarm-up 1/2: carregando {MODELO} na VRAM (aguarde até 5 min)...')
wu1 = {'model': MODELO, 'stream': False,
       'messages': [{'role': 'user', 'content': 'ok'}],
       'options': {'temperature': 0.1, 'num_ctx': 128, 'num_predict': 3}}
try:
    t_wu = time.time()
    req.post('http://localhost:11434/api/chat', json=wu1, timeout=360)
    print(f'  Carregamento: {time.time()-t_wu:.1f}s  |  VRAM: {vram_info()}')
    # 32B Q4 ocupa ~18 GB; A100 40GB deve mostrar ~18000 MiB usados
except Exception as e:
    sys.exit(f'Warm-up 1 falhou: {e}. Re-execute a Célula 5.')

# --- Warm-up 2: mede velocidade real de inferência ---
print('\nWarm-up 2/2: medindo velocidade real...')
wu2 = {'model': MODELO, 'stream': False,
       'messages': [{'role': 'user', 'content': 'Diga apenas "ok"'}],
       'options': {'temperature': 0.1, 'num_ctx': 256, 'num_predict': 20}}
try:
    t2 = time.time()
    r2 = req.post('http://localhost:11434/api/chat', json=wu2, timeout=60)
    lat2 = time.time() - t2
    tok2 = r2.json().get('eval_count', 0)
    toks2 = tok2 / lat2 if lat2 > 0 else 0
    print(f'  {tok2} tokens em {lat2:.1f}s → {toks2:.1f} tok/s')

    # Limiares para 32B na A100 (esperado ~12-18 tok/s)
    if toks2 >= 12:
        print(f'\n✓ A100 ativa ({toks2:.1f} tok/s). Esperado para 32B na A100.')
    elif toks2 >= 5:
        print(f'\n⚠ Velocidade abaixo do esperado ({toks2:.1f} tok/s).')
        print('  Pode ser L4 ou T4 em vez de A100. Verifique o tipo de ambiente.')
    else:
        print(f'\n✗ Muito lento ({toks2:.1f} tok/s) — modelo provavelmente na CPU.')
        print('  Reinicie e verifique LD_LIBRARY_PATH na Célula 5.')
except Exception as e:
    print(f'  Warm-up 2 falhou: {e}')

## Célula 7 — Função de recuperação (use se aparecer "Connection refused")

In [ ]:
import subprocess, time, requests as req, os, sys

def reiniciar_ollama():
    """
    Reinicia o servidor Ollama e recarrega o modelo na VRAM.
    Use apenas se aparecer 'Connection refused' durante a análise.
    """
    OLLAMA_BIN = '/usr/local/bin/ollama'
    subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
    time.sleep(2)

    env_rec = dict(os.environ)
    env_rec['OLLAMA_HOST']          = '0.0.0.0'
    env_rec['CUDA_VISIBLE_DEVICES'] = '0'
    env_rec['LD_LIBRARY_PATH']      = '/usr/lib64-nvidia:' + env_rec.get('LD_LIBRARY_PATH', '')

    subprocess.Popen(
        [OLLAMA_BIN, 'serve'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        env=env_rec
    )
    print('Servidor reiniciado. Aguardando...')
    for i in range(15):
        time.sleep(2)
        try:
            r = req.get('http://localhost:11434/api/tags', timeout=3)
            if r.status_code == 200:
                print(f'Ollama respondendo (tentativa {i+1}).')
                break
        except Exception:
            print(f'  Aguardando... ({i+1}/15)')
    else:
        sys.exit('Servidor não voltou. Reinicie o ambiente.')

    print(f'Recarregando {MODELO} na VRAM (aguarde ~5 min para 32B)...')
    req.post('http://localhost:11434/api/chat', json={
        'model': MODELO, 'stream': False,
        'messages': [{'role': 'user', 'content': 'ok'}],
        'options': {'num_ctx': 128, 'num_predict': 3}
    }, timeout=360)
    v = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print('VRAM:', v.stdout.strip())
    print('Pronto. Re-execute a célula de análise.')

print('✓ Função reiniciar_ollama() definida.')
print('  Para usar: chame reiniciar_ollama() em uma nova célula.')

## Célula 8 — Configurar e executar o agente

In [ ]:
import subprocess, time, os, re

os.chdir(WORK_DIR)

env = dict(os.environ)
env['PROVEDOR']             = 'ollama'
env['OLLAMA_MODELO']        = MODELO
env['OLLAMA_BASE_URL']      = 'http://localhost:11434'
env['OLLAMA_NUM_CTX']       = '16384'
env['OLLAMA_TIMEOUT']       = '360'
env['MAX_PROCESSOS']        = '10'    # altere para '0' para processar tudo
env['PYTHONUNBUFFERED']     = '1'     # força flush imediato de cada print/log
env['LD_LIBRARY_PATH']      = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
env['CUDA_VISIBLE_DEVICES'] = '0'

print('=' * 60)
print(f'Iniciando análise com {MODELO}...')
print(f'MAX_PROCESSOS = {env["MAX_PROCESSOS"]}  (0 = todas as licitações)')
print('=' * 60)

t_inicio = time.time()
proc = subprocess.Popen(
    ['python', 'pncp_agente.py'],
    cwd=str(WORK_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    encoding='utf-8',
    errors='replace',
)

re_prog = re.compile(r'\[(\d+)/(\d+)\]')

for linha in proc.stdout:
    linha = linha.rstrip()
    m = re_prog.search(linha)
    if m:
        atual, total = m.group(1), m.group(2)
        pct = int(atual) / int(total) * 100
        barra = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f'\n▶ [{barra}] {pct:5.1f}%  {linha}', flush=True)
    elif '[ERROR]' in linha or '[WARNING]' in linha:
        print(f'  ⚠  {linha}', flush=True)
    else:
        print(f'  {linha}', flush=True)

proc.wait()
t_total = time.time() - t_inicio

print(f'\n{"=" * 60}')
if proc.returncode == 0:
    print(f'✓ Concluído em {t_total/60:.1f} min.')
else:
    print(f'✗ Código {proc.returncode} após {t_total/60:.1f} min.')

## Célula 9 — Telemetria e benchmark

In [ ]:
import json, statistics
from pathlib import Path

tel_path = WORK_DIR / 'pncp_telemetria.jsonl'
if not tel_path.exists():
    print('pncp_telemetria.jsonl não encontrado — execute a Célula 8 primeiro.')
else:
    linhas = []
    with open(tel_path) as f:
        for linha in f:
            try: linhas.append(json.loads(linha))
            except Exception: pass

    llm    = [l for l in linhas if l.get('provedor') == 'ollama']
    pf     = [l for l in linhas if l.get('provedor') == 'pre_filtro']
    llm_ok = [l for l in llm if l.get('lat_ms') is not None]
    llm_err= [l for l in llm if l.get('lat_ms') is None]

    lats     = [l['lat_ms']  for l in llm_ok]
    toks_out = [l['tok_out'] for l in llm_ok if l.get('tok_out')]
    schema_ok= sum(1 for l in llm if l.get('schema_ok'))

    print(f'{"="*55}')
    print(f'BENCHMARK — {MODELO}')
    print(f'{"="*55}')
    print(f'Chamadas LLM totais:    {len(llm)}')
    print(f'  Com resposta:         {len(llm_ok)}')
    print(f'  Sem resposta (erro):  {len(llm_err)}')
    print(f'Pré-filtradas (sem LLM):{len(pf)}')
    print(f'Schema válido:          {schema_ok}/{len(llm)} ({schema_ok/len(llm)*100:.0f}%)')

    if lats:
        print(f'\nLatência (s):')
        print(f'  Mínima:   {min(lats)/1000:.1f}s')
        print(f'  Média:    {statistics.mean(lats)/1000:.1f}s')
        print(f'  Mediana:  {statistics.median(lats)/1000:.1f}s')
        print(f'  Máxima:   {max(lats)/1000:.1f}s')

    if toks_out and lats:
        tok_s = sum(toks_out) / (sum(lats) / 1000)
        print(f'\nTokens de saída:')
        print(f'  Média:    {statistics.mean(toks_out):.0f}')
        print(f'  Máximo:   {max(toks_out)}')
        print(f'  tok/s:    {tok_s:.1f}  (geração real ~15 tok/s + ~35s prefill por chamada)')

    if lats:
        procs_ok     = set(l.get('processo') for l in llm_ok)
        n_lic        = len(procs_ok)
        n_itens_tot  = sum(l.get('n_itens', 0) for l in llm_ok)
        tempo_s      = sum(lats) / 1000
        lic_h        = (n_lic / tempo_s) * 3600 if tempo_s > 0 else 0
        media_itens  = n_itens_tot / n_lic if n_lic else 0
        print(f'\nThroughput:')
        print(f'  Licitações únicas:      {n_lic}')
        print(f'  Média itens/licitação:  {media_itens:.1f}')
        print(f'  Tempo real inferência:  {tempo_s/60:.1f} min')
        print(f'  Licitações/hora:        {lic_h:.0f}')
        print(f'  Licitações/dia (24h):   {lic_h*24:.0f}')

    print(f'\n{"="*55}')
    print('Detalhes por chamada:')
    print(f'{"processo":<42} {"lote":<7} {"n":<4} {"lat_s":<8} {"tok_out":<8} {"tok/s":<7} {"schema"}')
    print('-' * 85)
    for l in llm:
        proc    = (l.get('processo') or '')[-38:]
        lat_ms  = l.get('lat_ms')
        tok_out = l.get('tok_out')
        lat_s   = f'{lat_ms/1000:.1f}s' if lat_ms is not None else 'N/A'
        ts      = f'{tok_out/(lat_ms/1000):.1f}' if (lat_ms and tok_out) else 'N/A'
        erro    = f' <- {str(l.get("erro",""))[:35]}' if l.get('erro') else ''
        print(f'{proc:<42} {str(l.get("lote","?")):<7} {l.get("n_itens",0):<4} '
              f'{lat_s:<8} {str(tok_out or "N/A"):<8} {ts:<7} '
              f'{"OK" if l.get("schema_ok") else "FAIL"}{erro}')

## Célula 10 — Classificações por licitação

In [ ]:
import json
from pathlib import Path
from collections import Counter

editais_dir = WORK_DIR / 'editais'
resultados  = []

for pasta in sorted(editais_dir.iterdir()):
    adh = pasta / 'aderencia.json'
    if not adh.exists():
        continue
    with open(adh) as f:
        r = json.load(f)
    if r.get('_pre_filtrado'):
        continue

    # Modo compacto: parecer_geral na raiz
    parecer  = r.get('parecer_geral', '?')
    contexto = ''
    stats    = r.get('estatisticas', {})

    resultados.append({
        'processo': pasta.name,
        'parecer':  parecer,
        'AD':  stats.get('aderencia_direta', 0),
        'PF':  stats.get('aderencia_parcial_forte', 0),
        'Pw':  stats.get('aderencia_parcial_fraca', 0),
        'FPL': stats.get('falso_positivo_lexical', 0),
        'NA':  stats.get('nao_aderente', 0),
        'itens': len(r.get('itens_analisados', [])),
    })

if not resultados:
    print('Nenhum resultado. Execute a Célula 8 primeiro.')
else:
    print(f'Licitações analisadas: {len(resultados)}\n')
    print(f'{"processo":<45} {"parecer":<25} {"AD":<4} {"PF":<4} {"Pw":<4} {"FPL":<5} {"NA":<4} {"itens"}')
    print('-' * 100)
    for r in resultados:
        print(f'{r["processo"][-42:]:<45} {r["parecer"]:<25} '
              f'{r["AD"]:<4} {r["PF"]:<4} {r["Pw"]:<4} {r["FPL"]:<5} {r["NA"]:<4} {r["itens"]}')

    pareceres = Counter(r['parecer'] for r in resultados)
    print(f'\nSumário: {dict(pareceres)}')

## Célula 11 — Download dos resultados

Gera `resultados_colab_a100.zip` com telemetria e todos os `aderencia.json`.

In [ ]:
from google.colab import files as colab_files
import zipfile

saida_zip = WORK_DIR / 'resultados_colab_a100.zip'

with zipfile.ZipFile(saida_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for nome in ['pncp_aderencias.json', 'pncp_telemetria.jsonl', 'pncp_itens_consolidado.json']:
        p = WORK_DIR / nome
        if p.exists():
            zf.write(p, nome)
            print(f'  + {nome}  ({p.stat().st_size / 1024:.0f} KB)')

    n_adh = 0
    for pasta in (WORK_DIR / 'editais').iterdir():
        adh = pasta / 'aderencia.json'
        if adh.exists():
            zf.write(adh, f'editais/{pasta.name}/aderencia.json')
            n_adh += 1
    print(f'  + editais/*/aderencia.json  ({n_adh} arquivos)')

print(f'\nFazendo download de {saida_zip.name}...')
colab_files.download(str(saida_zip))